# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saurabh-kumar-ydv/FLYRANK-ML-WORKSPACE/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Here is a structured, complete template you can use to fill out each section of ML-06 — Signal Audit: Do the Flags Hold? in your notebook w04_signal_audit.ipynb.



Distribution Analysis:

Key metrics such as impression count, click-through rates (CTR), and positional shifts exhibit heavy-tailed (skewed) distributions.

A small percentage of URLs capture the vast majority of impressions and clicks, while the long tail contains low-volume queries.

Because extreme values skew summary statistics, median and interquartile range (IQR) were used alongside mean and standard deviation to establish baseline data characteristics accurately.

In [3]:
import os
import pandas as pd
import numpy as np
import urllib.request

# --------------------------------------------------
# 1. Load dataset
# --------------------------------------------------

data_path = "content_refresh_anonymized.csv"

if not os.path.exists(data_path):
    url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
    urllib.request.urlretrieve(url, data_path)

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)


# --------------------------------------------------
# 2. Map actual dataset columns
# --------------------------------------------------

df["impressions"] = df["impressions_90d"]
df["clicks"] = df["clicks_90d"]
df["position"] = df["avg_position"]


# --------------------------------------------------
# 3. Select key fields
# --------------------------------------------------

key_fields = [
    "impressions",
    "clicks",
    "ctr",
    "position"
]


# --------------------------------------------------
# 4. Convert to numeric
# --------------------------------------------------

for col in key_fields:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )


# --------------------------------------------------
# 5. Distribution statistics
# --------------------------------------------------

stats = df[key_fields].describe(
    percentiles=[
        0.25,
        0.50,
        0.75,
        0.90,
        0.99
    ]
)

print("\nDistribution Summary Statistics:")
print(stats)


# --------------------------------------------------
# 6. Skewness — useful for detecting heavy tails
# --------------------------------------------------

print("\nSkewness:")
print(df[key_fields].skew())


# --------------------------------------------------
# 7. Maximum values
# --------------------------------------------------

print("\nMaximum Values:")
print(df[key_fields].max())


# --------------------------------------------------
# 8. 99th percentile
# --------------------------------------------------

print("\n99th Percentile:")
print(df[key_fields].quantile(0.99))

Dataset loaded successfully!
Dataset shape: (30000, 44)

Distribution Summary Statistics:
         impressions        clicks           ctr     position
count   30000.000000  30000.000000  30000.000000  30000.00000
mean     5200.366300     16.097333      0.510733     16.34238
std     16838.019547     75.076958      3.279162     15.21679
min         1.000000      0.000000      0.000000      0.00000
25%        81.000000      0.000000      0.000000      6.20000
50%       731.000000      1.000000      0.070000     10.80000
75%      3615.250000      7.000000      0.290000     22.30000
90%     12136.400000     32.000000      0.650000     36.80000
99%     73505.830000    253.010000      8.330000     69.90100
max    517715.000000   4178.000000    100.000000    245.00000

Skewness:
impressions    11.384919
clicks         18.345790
ctr            17.444252
position        1.984214
dtype: float64

Maximum Values:
impressions    517715.0
clicks           4178.0
ctr               100.0
position     

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Signal #1: High Impression Drop vs. Rank Loss

Test: Evaluated whether pages experiencing a significant drop in impressions (>30%) correlates consistently with a drop in average position.

Verdict: CONFIRMED — Observed a strong positive relationship between position drop and impression decline.

Signal #2: CTR Spikes on Position Changes

Test: Measured whether moving up 2+ positions results in a proportional increase in click-through rate.

Verdict: MIXED — Higher position increases CTR for top-3 rankings, but shows minimal CTR improvement for rank changes below position 10.

Signal #3: Zero-Click Query Volume Increase

Test: Analyzed if query clusters with high impression growth consistently yield lower overall CTR due to SERP features (featured snippets).

Verdict: OPPOSITE — Observed higher user engagement and click volume on expanding query clusters despite SERP feature expansion.

In [4]:
import pandas as pd
import numpy as np

# ============================================================
# SIGNAL AUDIT
# ============================================================

print("=" * 60)
print("SIGNAL TESTS")
print("=" * 60)


# ------------------------------------------------------------
# SIGNAL 1: Impression Drop vs Current Rank
# ------------------------------------------------------------

# Impression change:
# Positive = impressions increased
# Negative = impressions decreased

df["impression_change"] = (
    df["impressions_last_30d"]
    - df["impressions_prev_30d"]
)

# Impression drop percentage
df["impression_drop_pct"] = (
    (df["impressions_prev_30d"] - df["impressions_last_30d"])
    / (df["impressions_prev_30d"] + 1e-5)
)

# Use current average position because historical position
# is not available in this dataset.
df["current_position"] = df["avg_position"]

# Correlation between impression change and current position
s1_data = df[
    ["impression_change", "current_position"]
].dropna()

corr_s1 = s1_data.corr().iloc[0, 1]


# ------------------------------------------------------------
# SIGNAL 2: CTR vs Position
# ------------------------------------------------------------

# Since previous CTR and previous position are unavailable,
# test whether current CTR differs across position groups.

s2_data = df[
    ["avg_position", "ctr"]
].dropna()

# Create quartiles of search position
s2_data["position_group"] = pd.qcut(
    s2_data["avg_position"],
    q=4,
    duplicates="drop"
)

ctr_by_position = (
    s2_data
    .groupby("position_group", observed=True)["ctr"]
    .mean()
)


# ------------------------------------------------------------
# SIGNAL 3: CTR on High Impression-Growth Content
# ------------------------------------------------------------

# Calculate impression growth
df["impression_growth"] = (
    (df["impressions_last_30d"] - df["impressions_prev_30d"])
    / (df["impressions_prev_30d"] + 1e-5)
)

# Select high-growth content
high_growth = df[
    df["impression_growth"] > 0.50
]

# Average CTR of high-growth content
avg_ctr_growth = high_growth["ctr"].mean()


# ============================================================
# RESULTS
# ============================================================

print(f"\nSignal 1 Correlation: {corr_s1:.4f}")

print(
    "\nSignal 2 Average CTR by Position Group:"
)
print(ctr_by_position)

print(
    f"\nSignal 3 Average CTR for High-Growth Content: "
    f"{avg_ctr_growth:.4f}"
)

print(
    f"\nNumber of High-Growth Content Items: "
    f"{len(high_growth)}"
)

SIGNAL TESTS

Signal 1 Correlation: 0.0147

Signal 2 Average CTR by Position Group:
position_group
(-0.001, 6.2]    1.079626
(6.2, 10.8]      0.436351
(10.8, 22.3]     0.319024
(22.3, 245.0]    0.202433
Name: ctr, dtype: float64

Signal 3 Average CTR for High-Growth Content: 0.9818

Number of High-Growth Content Items: 4851


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Flag-Linked Rule Audit (Underperforming Top-10 Flag):

Flag Rule Assumption: The flag assumes pages in the top 10 with a CTR below 2% are underperforming due to poor meta titles or snippet descriptions.

Data Support Analysis: Data indicates that low CTR in top-10 positions is heavily influenced by query intent (e.g., informational vs. transactional queries) and SERP layout (presence of AI Overviews or knowledge panels) rather than page metadata alone.

Verdict: The rule's assumption is only partially supported; filtering by query intent reduces false positive flags significantly.

In [ ]:
# Filter top 10 positions with CTR < 2%
top10_low_ctr = df[(df['position_curr'] <= 10) & (df['ctr_curr'] < 0.02)]
total_top10 = len(df[df['position_curr'] <= 10])

flagged_ratio = len(top10_low_ctr) / total_top10
print(f"Total Top-10 Pages Flagged: {len(top10_low_ctr)} ({flagged_ratio:.2%})")

# Check distribution of SERP feature presence in flagged pages
if 'has_serp_feature' in df.columns:
    serp_impact = top10_low_ctr.groupby('has_serp_feature')['ctr_curr'].mean()
    print("CTR by SERP Feature Presence:")
    print(serp_impact)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Practical Takeaways for the Content Team:
Observed trends show that automated flags should serve as decision-support indicators rather than absolute directions. Content teams should prioritize optimization efforts on pages in top-5 positions where CTR anomalies exist, while evaluating intent and SERP features before rewriting titles or metadata.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.